<a href="https://colab.research.google.com/github/aliakseizvertouski/etl_task/blob/main/python_task_oop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Задача 1: Инкапсуляция и защищенные атрибуты

Название: Система обработки платежей (Base Gateway)

Условие: Создайте базовый класс PaymentGateway. Он должен хранить api_key. Доступ к ключу напрямую должен быть ограничен (используйте _ или __).

Реализуйте метод для «маскирования» ключа (показывать только первые 4 символа).

In [ ]:
class PaymentGateway:
    def __init__(self, api_key: str):
        self.__api_key = api_key


    def get_masked_key(self) -> str:

        masked_key = ""
        for i in range(len(self.__api_key)):
            if i < 4:
                masked_key += self.__api_key[i]
            else:
                masked_key += "*"

        return masked_key


gateway = PaymentGateway("sk_test_123456789")
assert gateway.get_masked_key() == "sk_t*************"  #нехватало звездочек
assert hasattr(gateway, "_PaymentGateway__api_key") or hasattr(gateway, "_api_key")




print (gateway.get_masked_key())

sk_t*************


### Задача 2: Использование @property для валидации

Название: Модель пользователя (User Profile)

Условие: Создайте класс UserProfile. Поле email должно быть доступно через property. При попытке установить email без символа @, должно выбрасываться исключение ValueError.

In [ ]:

class UserProfile:

    def __init__(self, email: str):
        self._email = None
        self.email = email

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, value: str):
        if '@' not in value:
            raise ValueError("Email must contain '@'")
        self._email = value



# Asserts
user = UserProfile("test@example.com")
assert user.email == "test@example.com"
try:
    user.email = "bad_email"
except ValueError:
    print("Validation works!")

Validation works!


### Задача 3: Наследование и типизация (Type Hinting)

Название: Иерархия ответов API

Условие: Создайте базовый класс BaseResponse с атрибутом status_code. Создайте подклассы SuccessResponse (код 200) и ErrorResponse (код 400). Используйте аннотации типов.

In [ ]:

from typing import Any

class BaseResponse:
    def __init__(self, status_code: int, data: Any) -> None:
        self.status_code: int = status_code
        self.data: Any = data

class SuccessResponse(BaseResponse):
    def __init__(self, data: Any) -> None:
        super().__init__(status_code=200, data=data)

class ErrorResponse(BaseResponse):
    def __init__(self, error_message: str) -> None:
        super().__init__(status_code=400, data=error_message)


s = SuccessResponse({"id": 1})
e = ErrorResponse("Not Found")

print (s.status_code)
print (e.status_code)
print (e.data)

assert s.status_code == 200
assert e.status_code == 400
assert e.data == "Not Found"

200
400
Not Found


### Задача 4: Множественное наследование и MRO

Название: Mixins для логирования и аутентификации

Условие: Создайте LoggingMixin с методом log(msg) и AuthMixin с методом authenticate(). Создайте класс SecureService, который наследует оба миксина. Проверьте порядок разрешения методов.

In [ ]:
# Task 4
class LoggingMixin:
    def log(self, message: str):
        return f"LOG: {message}"

class AuthMixin:
    def authenticate(self):
        return "Authenticated"

class SecureService(LoggingMixin, AuthMixin):
    pass


# SecureService.__mro__
print(SecureService.mro())


# Asserts
service = SecureService()
assert service.log("test") == "LOG: test"
assert service.authenticate() == "Authenticated"
assert SecureService.__mro__[1] == LoggingMixin

[<class '__main__.SecureService'>, <class '__main__.LoggingMixin'>, <class '__main__.AuthMixin'>, <class 'object'>]


### Задача 5: Приватные методы и Docstrings

Название: Внутренняя кухня парсера

Условие: Создайте класс DataParser. Реализуйте публичный метод parse(data). Внутри него должен вызываться приватный метод __clean_data(data). Обязательно напишите Google-style или Sphinx докстринг для класса.

In [ ]:
# Task 5
class DataParser:
    """
    Class for parsing raw data.

    Attributes:
        encoding (str): Data encoding.
    """
    def __init__(self, encoding: str = "utf-8"):
        self.encoding = encoding

    def __clean_data(self, data: str) -> str:
        """Strips whitespace and converts to lowercase."""
        return data.strip().lower()

    def parse(self, data: str) -> str:

        """
        Parses raw data.

        Args:
            data (str): Raw data.

        Returns:
            str: Cleaned data.

        """

        # """
        # parses raw data.

        # :param data: raw data
        # :type data: str

        # """

        return self.__clean_data(data)

print('класс DataParse', DataParser.__doc__)
print('метод parse: ', DataParser.parse.__doc__)

# Asserts
parser = DataParser()
assert parser.parse("  DATA  ") == "data"
assert not hasattr(parser, "__clean_data") # К методу нельзя обратиться напрямую


    Class for parsing raw data.

    Attributes:
        encoding (str): Data encoding.
    
метод parse:  
        Parses raw data.

        Args:
            data (str): Raw data.

        Returns:
            str: Cleaned data.
        
        


### Задача 6: Паттерн Фабрика и isinstance

Название: Фабрика уведомлений

Условие: Есть базовый класс Notification и наследники SMSNotification, EmailNotification. Создайте класс NotificationFactory с методом create(type: str). Проверьте типы созданных объектов через isinstance.

In [ ]:
# Task 6
class Notification: pass
class SMSNotification(Notification): pass
class EmailNotification(Notification): pass

class NotificationFactory:
    @staticmethod
    def create(notif_type: str) -> Notification:
        if notif_type == "sms":
            return SMSNotification()
        elif notif_type == "email":
            return EmailNotification()
        else:
            return None

# Asserts
factory = NotificationFactory()
sms = factory.create("sms")
email = factory.create("email")

# #Ой
# phone_number = factory.create("phone")
# print(isinstance(phone_number, SMSNotification))

assert isinstance(sms, SMSNotification)
assert isinstance(email, EmailNotification)
assert issubclass(SMSNotification, Notification)

### Задача 7: Паттерн Репозиторий + Dependency Injection

Название: Отвязка логики от способа хранения (User Storage)
Условие: 1. Создайте интерфейс (ABC) IUserRepository с методами add(user) и get_all().
2. Реализуйте две версии:

InMemoryUserRepository (хранит объекты в списке _users).

FileUserRepository (имитирует запись в файл — для простоты пусть просто печатает "Saving to disk..." и тоже хранит в списке).

Создайте класс UserService, который принимает любой репозиторий через __init__.

Сервис должен иметь метод register_user(name), который создает пользователя и сохраняет его через репозиторий.

Цель: Показать, что UserService работает одинаково с любым «бэкендом» данных.

In [ ]:
# Task 7 (Updated)
from abc import ABC, abstractmethod
from typing import List

class User:
    def __init__(self, name: str):
        self.name = name

class IUserRepository(ABC):
    @abstractmethod
    def add(self, user: User):
        pass

    @abstractmethod
    def get_all(self) -> List[User]:
        pass

class InMemoryUserRepository(IUserRepository):
    def __init__(self):
        self._users = []

    def add(self, user: User):
        self._users.append(user)

    def get_all(self) -> List[User]:
        return self._users

class FileUserRepository(IUserRepository):
    def __init__(self):
        self._users = []

    def add(self, user: User):
        print('Saving to disk...')
        self._users.append(user)

    def get_all(self) -> List[User]:
        return self._users

class UserService:
    def __init__(self, repo: IUserRepository):
        self.repo = repo

    def register_user(self, name: str):
        self.repo.add(User(name))

# Asserts
repo_memory = InMemoryUserRepository()
service = UserService(repo_memory)
service.register_user("Alice")
assert len(repo_memory.get_all()) == 1
assert repo_memory.get_all()[0].name == "Alice"

repo_file = FileUserRepository()
service_file = UserService(repo_file)
service_file.register_user("Bob")
assert len(repo_file.get_all()) == 1
assert isinstance(repo_file, IUserRepository)

Saving to disk...


а теперь файл

In [ ]:
from abc import ABC, abstractmethod
from typing import List

path = '/content/users.txt'
# names = ['Alice', 'Bob', 'Vasya', 'Petya']

class User:
    def __init__(self, name: str):
        self.name = name

    def __str__(self):
        return self.name

class IUserRepository(ABC):
    @abstractmethod
    def add(self, user: User):
        pass

    @abstractmethod
    def get_all(self):
        pass

class InMemoryUserRepository(IUserRepository):
    def __init__(self):
        self._users = []

    def add(self, user: User):
        print('adding users in ram...')
        self._users.append(user)

    def get_all(self) -> List[User]:
        print ('getting users from ram...')
        return self._users

class FileUserRepository(IUserRepository):
    def __init__(self, path):
        self.path = path

    def add(self, user: User):

        with open(self.path, "w") as file:
            print(f'Saving to disk:' '\n' f'{user.name}')
            add_user = file.write(str(user.name))


    def get_all(self):
        with open(self.path, "r") as file:
            print ('чтение...')

            self.content = []
            for i, row in enumerate(file):
                self.content.append(row.strip())
                print(f'арегистрированные пользователи: {self.content}')
            return self.content


class UserService:
    def __init__(self, repo: IUserRepository):
        self.repo = repo

    def register_user(self, name: str):
        self.repo.add(User(name))

# Asserts
repo_memory = InMemoryUserRepository()
service = UserService(repo_memory)
service.register_user("Alice")
assert len(repo_memory.get_all()) == 1
assert repo_memory.get_all()[0].name == "Alice"

repo_file = FileUserRepository(path) #добавлен path в аргументы
service_file = UserService(repo_file)
service_file.register_user("Bob")
assert len(repo_file.get_all()) == 1 # мы не можем передавать больше 1 имени в repo
assert isinstance(repo_file, IUserRepository)

adding users in ram...
getting users from ram...
getting users from ram...
Saving to disk:
Bob
чтение...
арегистрированные пользователи: ['Bob']


In [ ]:
from os import write

path = '/content/users.txt'
names = ['Alice', 'Bob', 'Vasya', 'Petya', 'Dasha']

def write_names_to_file(names, path):

    with open(path, "w") as file:
        print(f'запись:' '\n' f'{names}')

        count = file.write(str(len(names)) + '\n')

        for i in names:
            content = file.write(i + '\n')

    return content

def read_names_from_file(path):
    with open(path, "r") as file:
        print ('чтение...')

        # content = ''
        # for i in file:
        #     content += i.strip() + ' '


        # for i in range(int(file.readline())):
        #     content = file.readline().strip()
        #     print(content)

        # for i in file:
        #     print(i.strip())

        content = []
        for i, row in enumerate(file):
            if i > 0:
                content.append(row.strip())
        return content


write_names_to_file(names, path)
read_names_from_file(path)


assert read_names_from_file(path) == names

запись:
['Alice', 'Bob', 'Vasya', 'Petya', 'Dasha']
чтение...
чтение...


### Задача 8: Композиция вместо наследования



Название: Сервис и Репозиторий

Условие: Создайте класс UserService. Он не должен наследоваться от репозитория, а должен принимать его в __init__ (Dependency Injection). Реализуйте метод get_user_name.

In [ ]:
# Task 8

class BaseRepository:
    def __init__(self):
        self.storage = {}

    def get_user_name(self, user_id):
        return self.storage.get(user_id)

class UserService:
    def __init__(self, repository: BaseRepository):
        self.repository = repository


    def get_user_name(self, user_id: int) -> str:
        return self.repository.get_user_name(user_id)

class InMemoryRepository(BaseRepository):
    def save(self, user_id, name):
        self.storage[user_id] = name


# Asserts
repo = InMemoryRepository()
repo.save(42, "John Doe")
service = UserService(repo)
assert service.get_user_name(42) == "John Doe"

In [ ]:
storage = {}

storage[1] = 'Alex'
storage [2] = 'Gleb'
storage[3] = 'nikita'

name = storage[1]
print (storage)
print(name)

{1: 'Alex', 2: 'Gleb', 3: 'nikita'}
Alex


### Задача 9: Работа с MRO и super()

Название: Кастомный логгер транзакций

Условие: Создайте класс Transaction. Создайте класс LoggedTransaction, который переопределяет метод execute(), вызывает super().execute() и возвращает строку с припиской " (logged)".

In [ ]:
# Task 9
class Transaction:
    def execute(self):
        return "Action"

class LoggedTransaction(Transaction):
    def execute(self):

        return super().execute() + ' (logged)'

# Asserts
t = LoggedTransaction()
assert t.execute() == "Action (logged)"

### Interstellar Fleet Commander (Architecture Task)

Условие
Вам нужно разработать ядро системы управления космическими кораблями.

Основные требования:

Компоненты (Наследование и MRO): Есть базовый класс BaseModule и наследники: Engine, Shield, LifeSupport. Добавьте миксин StatusMixin для отслеживания износа.

Инкапсуляция (Private & Property): Корабль (SpaceShip) должен иметь приватный атрибут __fuel_level. Доступ к нему — через property с валидацией (топливо не может быть отрицательным или больше 100%).

Репозиторий (Pattern): Создайте ICargoRepository для управления грузами на борту. Реализуйте версию InMemoryCargo, которая хранит список названий грузов.

Фабрика и Типы (Factory & isinstance): Класс ModuleFactory должен создавать модули по строковому названию.

Валидатор (issubclass): Метод в системе должен проверять, можно ли установить деталь в слот, проверяя, что класс детали является подклассом BaseModule.

In [ ]:
from abc import ABC, abstractmethod
from typing import List, Any, Optional

# --- 1. Mixins & Hierarchy ---

class StatusMixin:
    """Миксин для добавления статуса износа (integrity)."""
    def get_integrity(self) -> str:
        return f"{self._integrity}%"

class BaseModule(ABC, StatusMixin):
    """Базовый класс для всех модулей корабля."""
    def __init__(self, model_name: str):
        self.model_name = model_name
        self._integrity = 100

class Engine(BaseModule):
    def __init__(self, model_name: str):
        super().__init__(model_name)

class Shield(BaseModule):
    def __init__(self, model_name: str):
        super().__init__(model_name)

'''
(@____@)
ну то есть грубо говоря, мы могли get_integrity создать в BaseModule, но решили его вынести отдельно.
так как она в аргументах BaseModule, наследники BaseModule могут использовать этот метод.
да?
'''

# --- 2. Repository Pattern ---

class ICargoRepository(ABC):
    """Интерфейс для работы с грузовым отсеком."""
    @abstractmethod
    def add_item(self, item: str):
        pass

    @abstractmethod
    def list_items(self) -> List[str]:
        pass

class InMemoryCargo(ICargoRepository):
    def __init__(self):
        self._storage = []

    def add_item(self, item: str):
        print (f'добавлен груз: {item}')
        self._storage.append(item)

    def list_items(self) -> List[str]:
        print (f'груз: {self._storage}')
        return self._storage

'''
ну это было изи
'''

# --- 3. Core Class (Encapsulation & Composition) ---

class SpaceShip:
    """
    Главный класс корабля.
    Использует композицию: содержит модули и репозиторий груза.
    """
    def __init__(self, name: str, cargo_repo: ICargoRepository):
        self.name = name
        self.cargo = cargo_repo
        self.modules: List[BaseModule] = []
        self.__fuel_level = 100.0 # Приватный атрибут

    @property
    def fuel(self) -> float:
        return self.__fuel_level

    @fuel.setter
    def fuel(self, value: float):
        """Установка топлива с валидацией (0-100)."""
        if value <0 or value >100:
            raise ValueError('валидация не пройдена')
        self.__fuel_level = value

    def install_module(self, module: Any):
        """
        Установка модуля.
        ВАЖНО: Проверять через isinstance, что это BaseModule.
        """
        if not isinstance(module, BaseModule):
            raise TypeError("не является BaseModule")
        else: self.modules.append(module)

# --- 4. Factory & Validation ---

class ModuleFactory:
    """Фабрика для создания модулей."""
    @staticmethod
    def build(module_type: str, model: str) -> BaseModule:
        """
        Создает Engine или Shield.
        Если тип неизвестен — raise ValueError.
        """

        if module_type.lower() == "engine":
            return Engine(model)
        elif module_type.lower() == "shield":
            return Shield(model)
        else:
            raise ValueError(f"Неизвестный модуль: {module_type}")

def check_compatibility(module_cls: Any) -> bool:
    """
    Проверяет, является ли переданный КЛАСС (не объект)
    валидным модулем для корабля (используйте issubclass).
    """
    return issubclass(module_cls, BaseModule)

Ассерты для проверки (запускать после реализации)

In [ ]:
# 1. Проверка Репозитория
repo = InMemoryCargo()
repo.add_item("Water")
repo.add_item("Oxygen Tanks")
assert len(repo.list_items()) == 2
assert "Water" in repo.list_items()

# 2. Проверка Инкапсуляции и Property
ship = SpaceShip(name="Discovery", cargo_repo=repo)
ship.fuel = 50.5
assert ship.fuel == 50.5
try:
    ship.fuel = 150 # Должно вызвать ошибку
except ValueError:
    print("Fuel validation: OK")

# 3. Проверка Фабрики и MRO
factory = ModuleFactory()
warp_engine = factory.build("engine", "Warp-DRV-9")
deflector = factory.build("shield", "Aegis-X")

assert isinstance(warp_engine, Engine)
assert isinstance(deflector, StatusMixin) # Проверка миксина
assert warp_engine.get_integrity() == "100%" # Если реализовали возврат строки

# 4. Проверка Установки и isinstance
ship.install_module(warp_engine)
ship.install_module(deflector)
assert len(ship.modules) == 2

try:
    ship.install_module("Not a module") # Должно упасть внутри метода
except TypeError:
    print("Module type check: OK")

# # 5. Проверка issubclass
assert check_compatibility(Engine) is True
assert check_compatibility(str) is False

print("\n--- ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО ---")

добавлен груз: Water
добавлен груз: Oxygen Tanks
груз: ['Water', 'Oxygen Tanks']
груз: ['Water', 'Oxygen Tanks']
Fuel validation: OK
Module type check: OK

--- ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО ---
